# S23DR 2026 — HSS Evaluation

Evaluates a trained `RoofWireframeNet` checkpoint on the validation split
using the **Hausdorff Segment Score (HSS)**.

**Steps**
1. Check GPU
2. Mount Google Drive
3. Install dependencies & pull latest repo code
4. Load checkpoint from Drive
5. Load the S23DR validation dataset
6. Run inference + compute HSS
7. Results summary & confidence threshold sweep

## 1 · Check GPU

In [ ]:
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU      : {props.name}  ({props.total_memory / 1e9:.1f} GB)")
    DEVICE = "cuda"
else:
    print("GPU      : not available — using CPU (will be slow)")
    DEVICE = "cpu"

print(f"Device   : {DEVICE}")

## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
CKPT_PATH = "/content/drive/MyDrive/s23dr_last.pt"   # ← change if needed
assert os.path.exists(CKPT_PATH), f"Checkpoint not found: {CKPT_PATH}"
print(f"Checkpoint found: {CKPT_PATH}  ({os.path.getsize(CKPT_PATH)/1e6:.1f} MB)")

## 3 · Install dependencies & pull latest code

In [ ]:
!pip install -q datasets huggingface_hub scipy numpy

In [ ]:
import os

REPO_DIR = "/content/3d_building_construction"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull origin main
else:
    !git clone https://github.com/12turtleships/3d_building_construction {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 4 · Load checkpoint

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import torch
from s23dr.model import RoofWireframeNet, WireframeLoss

device = torch.device(DEVICE)

try:
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(CKPT_PATH, map_location=device)

saved_args = ckpt.get("args", {})
N_QUERIES  = saved_args.get("n_queries", 64)
epoch      = ckpt.get("epoch", "?")
val_loss   = ckpt.get("val_loss", float("nan"))

print(f"epoch     = {epoch}")
print(f"val_loss  = {val_loss:.4f}")
print(f"n_queries = {N_QUERIES}")

model = RoofWireframeNet(n_queries=N_QUERIES).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")
print("Model loaded successfully.")

## 5 · Load the S23DR validation dataset

Streams from the public HF dataset — no local download needed.  
If you have the dataset saved locally via `save_to_disk`, set `USE_LOCAL = True`.

In [ ]:
# ── configuration ────────────────────────────────────────────────────────────────────────────
USE_LOCAL  = False                              # True → load from LOCAL_DATA_DIR
LOCAL_DATA_DIR = "/content/3d_building_construction/s23dr/data"
HF_DATASET = "usm3d/s23dr-2026-sampled_4096_v2"
SPLIT      = "validation"
N_POINTS   = 1024
BATCH_SIZE = 8
CONF_THRESH = 0.5   # vertex confidence threshold (tuned in step 7)
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import io, zipfile
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


def _unpack_row(row):
    blob = row["data"]
    out = {}
    with zipfile.ZipFile(io.BytesIO(blob)) as zf:
        for name in zf.namelist():
            if name.endswith(".npy"):
                out[name[:-4]] = np.load(io.BytesIO(zf.read(name)), allow_pickle=False)
    out["order_id"] = row.get("order_id", "")
    return out


class S23DRValDataset(Dataset):
    def __init__(self, rows, n_points=1024):
        self._rows = [_unpack_row(r) for r in rows]
        self.n_points = n_points

    def __len__(self):
        return len(self._rows)

    def __getitem__(self, idx):
        r = self._rows[idx]
        N   = min(self.n_points, len(r["xyz_norm"]))
        sel = np.arange(N)
        return {
            "order_id": r["order_id"],
            "xyz":       torch.from_numpy(r["xyz_norm"][sel]).float(),
            "vote_frac": torch.from_numpy(r["vote_frac"][sel]).float(),
            "n_views":   torch.from_numpy(r["n_views_voted"][sel].astype(np.float32)).float(),
            "mask":      torch.from_numpy(r["mask"][sel].astype(np.float32)).float(),
            "class_id":  torch.from_numpy(r["class_id"][sel].astype(np.int64)),
            "gt_segs":   torch.from_numpy(r["gt_segments"]).float(),  # (E, 2, 3)
        }


def _collate(batch):
    return {
        "order_id":  [b["order_id"]  for b in batch],
        "xyz":       torch.stack([b["xyz"]       for b in batch]),
        "vote_frac": torch.stack([b["vote_frac"] for b in batch]),
        "n_views":   torch.stack([b["n_views"]   for b in batch]),
        "mask":      torch.stack([b["mask"]      for b in batch]),
        "class_id":  torch.stack([b["class_id"]  for b in batch]),
        "gt_segs":   [b["gt_segs"].numpy()        for b in batch],
    }


# Load rows
if USE_LOCAL:
    from datasets import load_from_disk
    hf_ds = load_from_disk(LOCAL_DATA_DIR)
    rows  = list(hf_ds[SPLIT] if SPLIT in hf_ds else hf_ds)
else:
    from datasets import load_dataset
    rows = list(load_dataset(HF_DATASET, split=SPLIT))

val_ds = S23DRValDataset(rows, n_points=N_POINTS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, collate_fn=_collate)

print(f"Loaded {len(val_ds)} validation samples  (batch_size={BATCH_SIZE})")

In [ ]:
# ── Coordinate space diagnostic ─────────────────────────────────────────────────────────────────────────
# Prints min/max/mean of each array to confirm pred_pos and gt_segments
# live in the same coordinate space.

r0 = val_ds._rows[0]

print("── Raw dataset arrays (sample 0) ──")
for key in ("xyz_norm", "gt_vertices", "gt_segments"):
    arr = r0.get(key)
    if arr is None:
        print(f"  {key}: NOT FOUND")
    else:
        print(f"  {key}: shape={arr.shape}  "
              f"min={arr.min():.4f}  max={arr.max():.4f}  mean={arr.mean():.4f}")

_batch0 = val_ds[0]
with torch.no_grad():
    _out0 = model(
        _batch0["xyz"].unsqueeze(0).to(device),
        _batch0["vote_frac"].unsqueeze(0).to(device),
        _batch0["n_views"].unsqueeze(0).to(device),
        _batch0["mask"].unsqueeze(0).to(device),
        _batch0["class_id"].unsqueeze(0).to(device),
    )
pred_pos0 = _out0["pred_pos"][0].cpu().numpy()
print(f"\n  pred_pos (model output): shape={pred_pos0.shape}  "
      f"min={pred_pos0.min():.4f}  max={pred_pos0.max():.4f}  mean={pred_pos0.mean():.4f}")

## 6 · Diagnose model output, then run inference & compute HSS

In [ ]:
diag_batch = next(iter(val_loader))
with torch.no_grad():
    diag_out = model(
        diag_batch["xyz"].to(device),
        diag_batch["vote_frac"].to(device),
        diag_batch["n_views"].to(device),
        diag_batch["mask"].to(device),
        diag_batch["class_id"].to(device),
    )

confs = torch.sigmoid(diag_out["pred_conf"][0]).cpu()
print("── Vertex confidence (first sample) ──")
print(f"  min={confs.min():.3f}  max={confs.max():.3f}  mean={confs.mean():.3f}")
print(f"  active (>0.5): {(confs > 0.5).sum().item()}  active (>0.1): {(confs > 0.1).sum().item()}")

edge_cls = diag_out["edge_logits"][0].argmax(dim=-1).cpu()
no_edge_class = 10
frac_no_edge = (edge_cls == no_edge_class).float().mean().item()
print("\n── Edge predictions (first sample) ──")
print(f"  Fraction predicting 'no edge': {frac_no_edge:.4f}")

if frac_no_edge > 0.999:
    print("\n⚠ Model predicts 'no edge' for every pair.")
else:
    print("\n✓ Model is predicting some edges — proceeding to full eval.")

In [ ]:
import time
from s23dr.metrics import hss, decode_to_segments

loss_fn = WireframeLoss()


@torch.no_grad()
def evaluate(model, loader, conf_thresh):
    model.eval()
    results = []
    t0 = time.time()

    for i, batch in enumerate(loader):
        out = model(
            batch["xyz"].to(device),
            batch["vote_frac"].to(device),
            batch["n_views"].to(device),
            batch["mask"].to(device),
            batch["class_id"].to(device),
        )

        for b in range(batch["xyz"].shape[0]):
            verts, edges, _ = loss_fn.decode(
                out["pred_pos"][b],
                out["pred_conf"][b],
                out["edge_logits"][b],
                conf_thresh=conf_thresh,
            )
            pred_segs = decode_to_segments(verts, edges)
            gt_segs   = batch["gt_segs"][b]
            scores    = hss(pred_segs, gt_segs)
            results.append({"order_id": batch["order_id"][b], **scores})

        if (i + 1) % 10 == 0:
            mean_hss = np.mean([r["hss"] for r in results])
            print(f"  [{len(results):4d}/{len(loader.dataset)}]  "
                  f"mean_hss={mean_hss:.4f}  ({time.time()-t0:.1f}s)")

    return results


print(f"Evaluating with conf_thresh={CONF_THRESH} …")
eval_results = evaluate(model, val_loader, CONF_THRESH)
print(f"Done — {len(eval_results)} samples evaluated")

## 7 · Results summary & confidence threshold sweep

In [ ]:
import numpy as np

hss_scores  = np.array([r["hss"]       for r in eval_results])
prec_scores = np.array([r["precision"] for r in eval_results])
rec_scores  = np.array([r["recall"]    for r in eval_results])

print("=" * 45)
print(f"  conf_thresh : {CONF_THRESH}")
print(f"  Samples     : {len(hss_scores)}")
print("-" * 45)
print(f"  HSS         : {hss_scores.mean():.4f}  (std {hss_scores.std():.4f})")
print(f"  Precision   : {prec_scores.mean():.4f}")
print(f"  Recall      : {rec_scores.mean():.4f}")
print("=" * 45)

In [ ]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
sweep_results = []

print(f"{'thresh':>8}  {'HSS':>8}  {'Prec':>8}  {'Recall':>8}")
print("-" * 40)

for thresh in thresholds:
    res = evaluate(model, val_loader, conf_thresh=thresh)
    h = np.mean([r["hss"]       for r in res])
    p = np.mean([r["precision"] for r in res])
    rc = np.mean([r["recall"]   for r in res])
    sweep_results.append((thresh, h, p, rc))
    print(f"{thresh:>8.2f}  {h:>8.4f}  {p:>8.4f}  {rc:>8.4f}")

best = max(sweep_results, key=lambda x: x[1])
print("-" * 40)
print(f"Best conf_thresh = {best[0]}  →  HSS = {best[1]:.4f}")

In [ ]:
sorted_results = sorted(eval_results, key=lambda r: r["hss"])

print("10 worst samples:")
print(f"{'order_id':>30}  {'HSS':>8}  {'Prec':>8}  {'Recall':>8}")
print("-" * 60)
for r in sorted_results[:10]:
    print(f"{str(r['order_id']):>30}  {r['hss']:>8.4f}  "
          f"{r['precision']:>8.4f}  {r['recall']:>8.4f}")

In [ ]:
import json, os

out_path = "/content/drive/MyDrive/s23dr_eval_results.json"
with open(out_path, "w") as f:
    json.dump({
        "checkpoint": CKPT_PATH,
        "split": SPLIT,
        "conf_thresh": CONF_THRESH,
        "mean_hss":       float(hss_scores.mean()),
        "mean_precision": float(prec_scores.mean()),
        "mean_recall":    float(rec_scores.mean()),
        "per_sample": eval_results,
    }, f, indent=2)

print(f"Results saved to {out_path}")

In [ ]:
# ── 8 · Visualise point cloud + predicted vs GT wireframe ────────────────────
import plotly.graph_objects as go

SAMPLE_IDX = 0
VIS_THRESH = 0.10
MAX_PTS    = 2048

r        = val_ds._rows[SAMPLE_IDX]
xyz_np   = r["xyz_norm"]
gt_segs  = r["gt_segments"]

item = val_ds[SAMPLE_IDX]
with torch.no_grad():
    out = model(
        item["xyz"].unsqueeze(0).to(device),
        item["vote_frac"].unsqueeze(0).to(device),
        item["n_views"].unsqueeze(0).to(device),
        item["mask"].unsqueeze(0).to(device),
        item["class_id"].unsqueeze(0).to(device),
    )
verts, edges, _ = loss_fn.decode(
    out["pred_pos"][0], out["pred_conf"][0], out["edge_logits"][0],
    conf_thresh=VIS_THRESH,
)
pred_segs = decode_to_segments(verts, edges)

rng = np.random.default_rng(0)
vis_idx = rng.choice(len(xyz_np), min(MAX_PTS, len(xyz_np)), replace=False)
pc = xyz_np[vis_idx]

def _line_trace(segs, color, name, width=4):
    if len(segs) == 0:
        return go.Scatter3d(x=[], y=[], z=[], mode="lines",
                            line=dict(color=color, width=width), name=name)
    xs, ys, zs = [], [], []
    for s in segs:
        xs += [float(s[0, 0]), float(s[1, 0]), None]
        ys += [float(s[0, 1]), float(s[1, 1]), None]
        zs += [float(s[0, 2]), float(s[1, 2]), None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode="lines",
                        line=dict(color=color, width=width), name=name)

hss_score = hss(pred_segs, gt_segs)
verts_np = verts.cpu().numpy() if hasattr(verts, "cpu") else np.array(verts)

fig = go.Figure(data=[
    go.Scatter3d(x=pc[:,0], y=pc[:,1], z=pc[:,2], mode="markers",
                 marker=dict(size=1.5, color="royalblue", opacity=0.35), name="Point cloud"),
    _line_trace(gt_segs,   "limegreen", f"GT ({len(gt_segs)} segs)"),
    _line_trace(pred_segs, "red",        f"Pred ({len(pred_segs)} segs)"),
])
fig.update_layout(
    title=(f"Sample {SAMPLE_IDX} | thresh={VIS_THRESH} | "
           f"HSS={hss_score['hss']:.3f}  P={hss_score['precision']:.3f}  R={hss_score['recall']:.3f}"),
    scene=dict(aspectmode="data"),
    height=680, margin=dict(l=0, r=0, t=50, b=0),
)
fig.show()

## 9 · Dataset schema: all arrays inside each ZIP sample

| Array | Shape | Dtype | Description |
|-------|-------|-------|-------------|
| `xyz_norm` | (N, 3) | float32 | Normalised 3-D point positions. |
| `vote_frac` | (N,) | float32 | Per-point view-agreement score (0–1). |
| `n_views_voted` | (N,) | int/float | Number of SfM views per point. |
| `mask` | (N,) | bool/float | Valid-point indicator. |
| `class_id` | (N,) | int64 | Semantic class label (0–63). |
| `source` | (N,) | int/float | **Target-building mask** (1 = target house, 0 = neighbouring buildings). The most important field after `xyz_norm` — the pipeline filters to `source==1` before any geometry processing. |
| `visible_src` | (N,) | int/float | Camera provenance: index of the source image that contributed this point. |
| `visible_id` | (N,) | int/float | Per-view point identifier. |
| `behind` | (N,) | int/float | Depth-layer index (0 = frontmost visible surface). |
| `gt_vertices` | (V, 3) | float32 | GT wireframe vertices in **world space** (metres). |
| `gt_edges` | (E, 2) | int64 | Edge connectivity: pairs of indices into `gt_vertices`. |
| `gt_edge_classes` | (E,) | int64 | Semantic edge type (0–9). |
| `gt_segments` | (E, 2, 3) | float32 | Pre-computed segments in **normalised** space — used directly by HSS. |
| `scale` | scalar | float32 | World = normalised × scale. |
| `center` | (3,) | float32 | World = normalised × scale + center. |

In [ ]:
import io, zipfile
import numpy as np

raw_row = rows[0]
blob = raw_row["data"]

print(f"ZIP byte length : {len(blob):,}")
print(f"order_id        : {raw_row['order_id']}")
print()
print(f"{'Array':25s}  {'Shape':18s}  {'Dtype':10s}  {'Min':>10s}  {'Max':>10s}  {'Mean':>10s}")
print("-" * 90)

with zipfile.ZipFile(io.BytesIO(blob)) as zf:
    for name in sorted(zf.namelist()):
        arr = np.load(io.BytesIO(zf.read(name)), allow_pickle=False)
        key = name.replace(".npy", "")
        mn  = float(arr.min())  if arr.size > 0 else float("nan")
        mx  = float(arr.max())  if arr.size > 0 else float("nan")
        mu  = float(arr.mean()) if arr.size > 0 else float("nan")
        print(f"{key:25s}  {str(arr.shape):18s}  {str(arr.dtype):10s}  "
              f"{mn:>10.4f}  {mx:>10.4f}  {mu:>10.4f}")

## 10 · S23DR 2026 — Competition Study

See `colab_procedural.ipynb` for the geometry-only baseline (HSS=0.5052, beats 2025 winner 0.43).

### Improvement roadmap

1. **Tighten edge selection** (highest impact): add minimum edge length + planarity-support filter to `s23dr/procedural/` to raise Precision from 0.49 → ~0.65, pushing HSS from 0.50 → ~0.67.
2. **Train longer / bigger**: increase `n_queries` (64→128), use `n_points=4096`, cosine LR warm-up, 200+ epochs on `sampled_4096_v3`.
3. **Upgrade backbone**: replace PointNet with PointNext or Point Transformer v3.
4. **Delaunay candidate graph**: use as edge prior (arXiv 2604.02497).
5. **Cylindrical-patch edge prediction** (2025 winner approach).
6. **Data augmentation**: random rotation, scale jitter, point dropout.